In [33]:
# libraries for text cleaning
%pip install nltk
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords as sw
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer

# PDF reader
%pip install pdfplumber
import pdfplumber

# convert date string to datetime format
from datetime import datetime

# data cleaning
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# file path checking
import os

# google robot to load data into google sheet and tableau
%pip install gspread oauth2client
import gspread
from oauth2client.service_account import ServiceAccountCredentials

from nltk.util import ngrams
from collections import Counter

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/henrytran/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/henrytran/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/henrytran/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 1, Extract the data 

In [34]:
# We should use this code
# this is the text extracted from the first 3 pages extracted from the
def raw_data_extraction(input_pdf):
    # take the pdf as an input and return the whole pdf as text
    raw_input=""
    with pdfplumber.open(input_pdf) as pdf:
        for page in pdf.pages:
            raw_input+=(page.extract_text() or "") + "\n"
    return raw_input

def article_extraction(raw_input):
    article=""
    details=""
    article_idx=0
    article_dict={}
    details_dict={}
    article_status=False # mark the start of the article
    details_status=False
    for row in raw_input.lower().split('\n'):
        if row=='full text':
            article_status=True
            article_idx+=1
            continue
        if article_status:
            if row=='details':
                article_status=False
                details_status=True
                article_dict[article_idx] = article
                article=""
                continue
            article+=row+" "
        if details_status:
            if row=='links':
                details_status=False
                details_dict[article_idx]=details
                details=""
                continue
            details+=row+'\n'
    return article_dict, details_dict

def clean_article(article):
    lemmatizer=WordNetLemmatizer()
    desc = re.sub(r'\([^)]*\)', '', article)
    sent_desc =sent_tokenize(desc)
    sww=set(sw.words())
    abstraction = []
    for sent in sent_desc:
        # normalise U.S., US, USA, U.S.A., etc. all to "america" (for classification)
        sent = re.sub(r"\b(?:u\.?\s*s\.?|usa|u\.?\s*s\.?\s*a\.?)\b", "america", sent, flags=re.IGNORECASE)
        # normalise to lowercase and only letters and numbers
        tokens = re.sub(r"[^a-z0-9]+", " ", sent.lower())
        words = word_tokenize(tokens)
        remaining_words = [word for word in words if word not in sww]
        if remaining_words:
            lemmatised_words=[]
            for word in remaining_words:
                lemmatised_word=lemmatizer.lemmatize(word)
                lemmatised_words.append(lemmatised_word)
            abstraction.append(lemmatised_words)
    final_doc=""
    for sent in abstraction[:-1]:
        for word in sent:
            final_doc+=word+" "
    return final_doc # remove the name of the writer

def clean_detail(detail):
    year=None
    date=None
    article_id=""
    for row in detail.split('\n'):
        # need to make this condition smarter
        label = re.sub(r'[^a-zA-Z0-9]', '', row.split(":")[0]).lower()
        if label == "proquestdocumentid":
            article_id=row.split(":")[1].strip().split(" ")
            if len(article_id)>1:
                new_id=""
                for part in article_id:
                    new_id+=part
                article_id=[new_id]
            continue
        #if row.split(":")[0]=="publication year":
            #year = str(row.split(":")[1].strip())
            #print(year)
        if label=="publicationdate":
            raw_date=row.split(":")[1].split(",")[0].strip().split(" ")
            raw_year=row.split(":")[1].split(",")[1].strip().split(" ")
            raw_date1=""
            raw_date2=""
            for value in raw_date: # add date
                raw_date1+=value
            for value in raw_year: # add year
                raw_date1+=value
            if len(raw_date1)==8:
                raw_date2+=raw_date1[:3]+" "+raw_date1[3]+" "+raw_date1[4:]
            if len(raw_date1)==9:
                raw_date2+=raw_date1[:3]+" "+raw_date1[3:5]+" "+raw_date1[5:]
            #full_raw_date= raw_date2 + " "+ year
            format_date= "%b %d %Y"           
            date = datetime.strptime(raw_date2, format_date)
    return (date.date() if date else None), article_id
    



In [35]:
import os
import json
import pandas as pd
from datetime import datetime

def process_all_pdfs(root_folder, manifest_path):
    """
    Iterates through all PDF folders, extracts text/details, 
    and returns dictionaries ready for phrase counting.
    """
    # 1. Load manifest to skip already processed files
    if not os.path.exists(root_folder):
        print(f"Error: The directory {root_folder} was not found. Check your file path!")
        return {}, {}
    
    if os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            manifest = json.load(f)
    else:
        manifest = {}

    all_new_articles = {}  # key: article_idx, value: lemmatized_text
    all_new_details = {}   # key: article_idx, value: (date, [proquest_id])
    
    # Global counter for this run to keep keys unique
    global_idx = 0

    # 2. Walk through the year folders
    for root, dirs, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith(".pdf"):
                file_path = os.path.join(root, file)
                
                # Check manifest by filename as a primary filter
                # (We will check ProQuest ID inside the article loop later)
                if file in manifest and manifest[file].get("status") == "completed":
                    print(f"Skipping: {file} (Already in manifest)")
                    continue

                print(f"Processing: {file}...")
                
                try:
                    # Apply your functions
                    raw_text = raw_data_extraction(file_path)
                    article_dict, details_dict = article_extraction(raw_text)
                    
                    # Clean and store
                    for idx in article_dict.keys():
                        # Extract and Clean
                        lemmatized_text = clean_article(article_dict[idx])
                        date_obj, proquest_id_list = clean_detail(details_dict[idx])
                        
                        # Use a unique index for this run
                        global_idx += 1
                        all_new_articles[global_idx] = lemmatized_text
                        all_new_details[global_idx] = (date_obj, proquest_id_list)
                    
                    # Update manifest for the file level
                    manifest[file] = {
                        "last_processed": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        "status": "completed"
                    }
                    
                except Exception as e:
                    print(f"Error processing {file}: {e}")
                    manifest[file] = {"status": "error", "error_msg": str(e)}

    # 3. Save updated manifest
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=4)

    return all_new_articles, all_new_details


In [51]:
# --- HOW TO RUN ---
root_path = "./data_articles/2016"
manifest_path = "./manifest.json"
new_articles, new_details = process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2016_4_2.pdf...
Processing: ProQuestDocuments-2016_4_1.pdf...
Processing: ProQuestDocuments-2016_14.pdf...
Processing: ProQuestDocuments-2016_2.pdf...
Processing: ProQuestDocuments-2016_3.pdf...
Processing: ProQuestDocuments-2016_1.pdf...
Processing: ProQuestDocuments-2016_12.pdf...
Processing: ProQuestDocuments-2016_4.pdf...
Error processing ProQuestDocuments-2016_4.pdf: not enough values to unpack (expected 1, got 0)
Processing: ProQuestDocuments-2016_5.pdf...
Processing: ProQuestDocuments-2016_13.pdf...
Processing: ProQuestDocuments-2016_11.pdf...
Processing: ProQuestDocuments-2016_7.pdf...
Processing: ProQuestDocuments-2016_6.pdf...
Processing: ProQuestDocuments-2016_10.pdf...
Processing: ProQuestDocuments-2016_8.pdf...
Processing: ProQuestDocuments-2016_9.pdf...


In [55]:
root_path = "./data_articles/2015"
manifest_path = "./manifest.json"
new_articles_15, new_details_15 = process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2015_3.pdf...
Processing: ProQuestDocuments-2015_2.pdf...
Processing: ProQuestDocuments-2015_1.pdf...
Processing: ProQuestDocuments-2015_5.pdf...
Processing: ProQuestDocuments-2015_4.pdf...
Processing: ProQuestDocuments-2015_6.pdf...
Processing: ProQuestDocuments-2015_7.pdf...
Processing: ProQuestDocuments-2015_13.pdf...
Processing: ProQuestDocuments-2015_12.pdf...
Processing: ProQuestDocuments-2015_9.pdf...
Processing: ProQuestDocuments-2015_10.pdf...
Processing: ProQuestDocuments-2015_11.pdf...
Processing: ProQuestDocuments-2015_8.pdf...
Processing: ProQuestDocuments-2015_15.pdf...
Processing: ProQuestDocuments-2015_14.pdf...
Processing: ProQuestDocuments-2015_16.pdf...


In [58]:
root_path = "./data_articles/2017"
manifest_path = "./manifest.json"
new_articles_17, new_details_17 = process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2017_1.pdf...
Processing: ProQuestDocuments-2017_2.pdf...
Processing: ProQuestDocuments-2017_3.pdf...
Processing: ProQuestDocuments-2017_7.pdf...
Processing: ProQuestDocuments-2017_6.pdf...
Processing: ProQuestDocuments-2017_4.pdf...
Processing: ProQuestDocuments-2017_5.pdf...
Processing: ProQuestDocuments-2017_61.pdf...
Processing: ProQuestDocuments-2017_8.pdf...
Processing: ProQuestDocuments-2017_9.pdf...
Processing: ProQuestDocuments-2017_10.pdf...
Processing: ProQuestDocuments-2017_11.pdf...
Processing: ProQuestDocuments-2017_13.pdf...
Processing: ProQuestDocuments-2017_12.pdf...


In [60]:
root_path = "./data_articles/2018"
manifest_path = "./manifest.json"
new_articles_18, new_details_18 = process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2018_11.pdf...
Processing: ProQuestDocuments-2018_10.pdf...
Processing: ProQuestDocuments-2018_12.pdf...
Processing: ProQuestDocuments-2018_13.pdf...
Processing: ProQuestDocuments-2018_16.pdf...
Processing: ProQuestDocuments-2018_14.pdf...
Processing: ProQuestDocuments-2018_8.pdf...
Processing: ProQuestDocuments-2018_9.pdf...
Processing: ProQuestDocuments-2018_15.pdf...
Processing: ProQuestDocuments-2018_4.pdf...
Processing: ProQuestDocuments-2018_5.pdf...
Processing: ProQuestDocuments-2018_7.pdf...
Processing: ProQuestDocuments-2018_6.pdf...
Processing: ProQuestDocuments-2018_2.pdf...
Processing: ProQuestDocuments-2018_3.pdf...
Processing: ProQuestDocuments-2018_1.pdf...


In [62]:
root_path = "./data_articles/2019"
manifest_path = "./manifest.json"
new_articles_19, new_details_19 = process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2019_8.pdf...
Processing: ProQuestDocuments-2019_9.pdf...
Processing: ProQuestDocuments-2019_13.pdf...
Processing: ProQuestDocuments-2019_7.pdf...
Processing: ProQuestDocuments-2019_6.pdf...
Processing: ProQuestDocuments-2019_12.pdf...
Processing: ProQuestDocuments-2019_10.pdf...
Processing: ProQuestDocuments-2019_4.pdf...
Processing: ProQuestDocuments-2019_5.pdf...
Processing: ProQuestDocuments-2019_11.pdf...
Processing: ProQuestDocuments-2019_15.pdf...
Processing: ProQuestDocuments-2019_1.pdf...
Processing: ProQuestDocuments-2019_14.pdf...
Processing: ProQuestDocuments-2019_16.pdf...
Processing: ProQuestDocuments-2019_2.pdf...
Processing: ProQuestDocuments-2019_3.pdf...
Processing: ProQuestDocuments-2019_17.pdf...


In [63]:
root_path = "./data_articles/2020"
manifest_path = "./manifest.json"
new_articles_20, new_details_20 = process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2020_9.pdf...
Processing: ProQuestDocuments-2020_8.pdf...
Processing: ProQuestDocuments-2020_13.pdf...
Processing: ProQuestDocuments-2020_12.pdf...
Processing: ProQuestDocuments-2020_10.pdf...
Processing: ProQuestDocuments-2020_11.pdf...
Processing: ProQuestDocuments-2020_3.pdf...
Processing: ProQuestDocuments-2020_2.pdf...
Processing: ProQuestDocuments-2020_1.pdf...
Processing: ProQuestDocuments-2020_5.pdf...
Processing: ProQuestDocuments-2020_4.pdf...
Processing: ProQuestDocuments-2020_6.pdf...
Processing: ProQuestDocuments-2020_7.pdf...


In [64]:
root_path = "./data_articles/2021"
manifest_path = "./manifest.json"
new_articles_21, new_details_21= process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2021_9.pdf...
Processing: ProQuestDocuments-2021_8.pdf...
Processing: ProQuestDocuments-2021_1.pdf...
Processing: ProQuestDocuments-2021_3.pdf...
Processing: ProQuestDocuments-2021_2.pdf...
Processing: ProQuestDocuments-2021_6.pdf...
Processing: ProQuestDocuments-2021_7.pdf...
Processing: ProQuestDocuments-2021_5.pdf...
Processing: ProQuestDocuments-2021_4.pdf...
Processing: ProQuestDocuments-2021_14.pdf...
Processing: ProQuestDocuments-2021_11.pdf...
Processing: ProQuestDocuments-2021_10.pdf...
Processing: ProQuestDocuments-2021_12.pdf...
Processing: ProQuestDocuments-2021_13.pdf...


In [71]:
root_path = "./data_articles/2022_01-2022_08"
manifest_path = "./manifest.json"
new_articles_2201_2208, new_details_2201_2208= process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2022-08_8.pdf...
Processing: ProQuestDocuments-2022-08_1.pdf...
Processing: ProQuestDocuments-2022-08_2.pdf...
Processing: ProQuestDocuments-2022-08_3.pdf...
Processing: ProQuestDocuments-2022-08_7.pdf...
Processing: ProQuestDocuments-2022-08_6.pdf...
Processing: ProQuestDocuments-2022-08_4.pdf...
Processing: ProQuestDocuments-2022-08_5.pdf...


In [72]:
root_path = "./data_articles/2022_08-2023_07"
manifest_path = "./manifest.json"
new_articles_2208_2307, new_details_2208_2307= process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-22-23_8.pdf...
Processing: ProQuestDocuments-22-23_9.pdf...
Processing: ProQuestDocuments-22-23_1.pdf...
Processing: ProQuestDocuments-22-23_10.pdf...
Processing: ProQuestDocuments-22-23_11.pdf...
Processing: ProQuestDocuments-22-23_2.pdf...
Processing: ProQuestDocuments-22-23_13.pdf...
Processing: ProQuestDocuments-22-23_12.pdf...
Processing: ProQuestDocuments-22-23_3.pdf...
Processing: ProQuestDocuments-22-23_7.pdf...
Processing: ProQuestDocuments-22-23_6.pdf...
Processing: ProQuestDocuments-22-23_4.pdf...
Processing: ProQuestDocuments-22-23_5.pdf...


In [73]:
root_path = "./data_articles/2023_08-2024_07"
manifest_path = "./manifest.json"
new_articles_2308_2407, new_details_2308_2407= process_all_pdfs(root_path, manifest_path)

Processing: ProQuest Documents 23-24_8.pdf...
Processing: ProQuest Documents 23-24_10.pdf...
Processing: ProQuest Documents 23-24_9.pdf...
Processing: ProQuest Documents 23-24_2.pdf...
Processing: ProQuest Documents 23-24_3.pdf...
Processing: ProQuest Documents 23-24_1.pdf...
Processing: ProQuest Documents 23-24_4.pdf...
Processing: ProQuest Documents 23-24_5.pdf...
Processing: ProQuest Documents 23-24_7.pdf...
Processing: ProQuest Documents 23-24_6.pdf...


In [74]:
root_path = "./data_articles/2024_08-2025"
manifest_path = "./manifest.json"
new_articles_2408_25, new_details_2408_25= process_all_pdfs(root_path, manifest_path)

Processing: ProQuestDocuments-2025-08-16_11.pdf...
Processing: ProQuestDocuments-2025-08-16_10.pdf...
Processing: ProQuestDocuments-2025-08-16_2.pdf...
Processing: ProQuestDocuments-2025-08-16_3.pdf...
Processing: ProQuestDocuments-2025-08-16_1.pdf...
Processing: ProQuestDocuments-2025-08-16_4.pdf...
Processing: ProQuestDocuments-2025-08-16_5.pdf...
Processing: ProQuestDocuments-2025-08-16_7.pdf...
Processing: ProQuestDocuments-2025-08-16_6.pdf...
Processing: ProQuestDocuments-2025-08-18_4.pdf...
Processing: ProQuestDocuments-2025-08-18_5.pdf...
Processing: ProQuestDocuments-2025-08-16_8.pdf...
Processing: ProQuestDocuments-2025-08-16_9.pdf...
Processing: ProQuestDocuments-2025-08-18_2.pdf...
Processing: ProQuestDocuments-2025-08-18_3.pdf...
Processing: ProQuestDocuments-2025-08-18_1.pdf...


## 2. Define the groups

In [39]:
# This is the new set of categories and terms used for classification, which is more comprehensive and detailed than the old one - can be modify if needed
groups = {
    'fiscal_terms':{
        'revenue_taxation': ['tax', 'taxation', 'taxed', 'subsidy', 'public sector revenue', 'import tariffs', 'import duty', 'trade act'],

        'government_spending': ['government spending', 'federal budget', 'defense spending', 'military spending', 
                                'public investment', 'social protection', 'healthcare expenditure', 'medical expenditure', 'pension reform'],
        
        'debt_borrowing': ['federal debt', 'public debt', 'national debt',
                            'debt ceiling', 'sovereign debt', 'government bond', 
                            'sovereign yield', 'borrowing', 'debt sustainability', 'debt default'],
        
        'budget_balance': ['budget deficit' , 'fiscal stimulus', 'balanced budget', 'budget gap', 
                            'national deficit', 'fiscal policy', 'public finance', 'debt consolidation',
                            'budget battle', 'fiscal footing', 'gramm-rudman'],
        
        'social_safety_net': ['social security', 'entitlement spending', 'social safety net', 'pension expenditure', 'medical care expenditure',
                                'medicaid', 'medicare', 'government welfare', 'food stamps', 'eitc']
    },

    'trade_terms':{
        'trade_policy': ['import tariffs', 'import duty', 'import barrier','government subsidy', 
                        'wto', 'world trade organization', 'trade treaty', 'trade agreement', 'trade policy', 'trade act', 
                        'doha round', 'uruguay round', 'gatt, dumping']
    },

    'tariff_terms': {
        'tariff_terms': ['tariffs','trade policy','trade disputes','international trade',
                  'trade relations','exports','supply chains','border security',
                    'immigration policy','immigration','deportation']
    },

    'america':{
        'america': ['america','american',
        
                    'white house', 'capitol hill', 'congress', 'senate', 'federal reserve', 'the fed', 'treasury department',

                    "alabama","alaska","arizona","arkansas","california","colorado",
                    "connecticut","delaware","florida","georgia","hawaii","idaho",
                    "illinois","indiana","iowa","kansas","kentucky","louisiana",
                    "maine","maryland","massachusetts","michigan","minnesota",
                    "mississippi","missouri","montana","nebraska","nevada",
                    "new hampshire","new jersey","new mexico","new york",
                    "north carolina","north dakota","ohio","oklahoma","oregon",
                    "pennsylvania","rhode island","south carolina","south dakota",
                    "tennessee","texas","utah","vermont","virginia","washington",
                    "west virginia","wisconsin","wyoming"]
    }
}

In [40]:
lemmatizer = WordNetLemmatizer()

def lemmatize_string(string, lemmatizer):
    return " ".join([lemmatizer.lemmatize(w.lower()) for w in string.split()])


def lemmatize_groups(groups, lemmatizer):
    new_groups={}
    for group_key, cats in groups.items():
        new_groups[group_key]={}
        for cat_key, terms in cats.items():
                lemmatized_terms = [lemmatize_string(term, lemmatizer) for term in terms]
                new_groups[group_key][cat_key] = lemmatized_terms
    return new_groups

In [41]:
lemmatized_groups=lemmatize_groups(groups, lemmatizer)

## 3. Count the matching groups and terms

In [68]:
import json
import os
import re
import pandas as pd
from datetime import datetime

# Define your file paths
MANIFEST_PATH = "manifest_wordcloud.json"
OUTPUT_CSV_PATH = "top_phrases_overtime.csv"
OUTPUT_JSON_CACHE = "wordcloud_processed_articles.json"

def load_manifest():
    """Loads the list of already processed article IDs or filenames."""
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {} # Return empty dictionary if manifest doesn't exist yet

def save_manifest(manifest_data):
    """Saves the updated checkpoint list to manifest.json."""
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest_data, f, indent=4)

def count_lemmatized_phrases_pipeline(lemmatized_groups, new_articles_dict, details_dict):
    """
    Processes only NEW articles, updates the manifest, and appends data 
    to the master phrase CSV for the Power BI Treemap.
    """
    # 1. Load the checkpoint tracking data
    manifest = load_manifest()
    
    # Flatten your lemmatized dictionary for faster lookup loops
    target_phrases = []
    for group_key, categories in lemmatized_groups.items():
        for category_key, terms in categories.items():
            for term in terms:
                # Expecting your input dictionary terms are already lemmatized (e.g., "government spend")
                target_phrases.append((group_key, category_key, term.lower()))

    new_phrase_records = []
    
    # 2. Loop through your freshly extracted text blocks
    for article_index, lemmatized_text in new_articles_dict.items():
            
        article_date = details_dict.get(article_index, [None])[0]
        article_id = details_dict.get(article_index, [None, None])[1]  # Assuming article_id is the second element in the list
        if not lemmatized_text or not article_date or not article_id:
            continue

        # Manifest Check: Skip if this specific article ID was already processed in a previous run
        if str(article_id) in manifest:
            continue
            
        text_lower = lemmatized_text.lower()
        article_has_matches = False

        # 3. Scan for exact lemmatized phrase combinations
        for group_key, category_key, lemmatized_phrase in target_phrases:
            
            # Using regex word boundaries (\b) to keep accuracy intact for lemmatized text
            matches = len(re.findall(r'\b' + re.escape(lemmatized_phrase) + r'\b', text_lower))
            
            if matches > 0:
                article_has_matches = True
                # Creating 1 distinct record per phrase match mapping
                new_phrase_records.append({
                    "Article_ID": article_id,
                    "Date": article_date,
                    "Classification_Group": group_key,
                    "Category": category_key,
                    "Phrase": lemmatized_phrase,  # e.g., "import tariff" instead of "import tariffs"
                    "Count": matches
                })
        
        # 4. Update manifest data memory structure for this article
        manifest[str(article_id)] = {
            "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "had_matches": article_has_matches
        }

    # 5. Save results and update files if new data was found
    if new_phrase_records:
        df_new = pd.DataFrame(new_phrase_records)
        df_new_aggregated = df_new.groupby(["Date", "Phrase", "Classification_Group"]).agg({"Count": "sum"}).reset_index()
        df_new_aggregated = df_new_aggregated[df_new_aggregated["Classification_Group"].str.lower() != "america"]
        
        # If the master CSV already exists, append to it; otherwise, create it
        if os.path.exists(OUTPUT_CSV_PATH):
            df_existing = pd.read_csv(OUTPUT_CSV_PATH)
            df_existing = df_existing[df_existing["Classification_Group"].str.lower() != "america"]  # Exclude "america" group from existing data to avoid duplication
            df_combined = pd.concat([df_existing, df_new_aggregated]).groupby(["Date", "Phrase", "Classification_Group"]).agg({"Count": "sum"}).reset_index()
            #df_combined = df_combined[df_combined["Classification_Group"].str.lower() != "america"]
            df_combined.to_csv(OUTPUT_CSV_PATH, index=False)
        else:
            df_new_aggregated = df_new_aggregated[df_new_aggregated["Classification_Group"].str.lower() != "america"] 
            df_new_aggregated.to_csv(OUTPUT_CSV_PATH, index=False) 
            
        # Update manifest checkpoint file on disk
        save_manifest(manifest)
        print(f"Successfully processed {len(new_phrase_records)} new phrase metrics and updated manifest.json.")
    else:
        print("No new articles to process. Dashboard data is up to date.")

    return OUTPUT_CSV_PATH

In [78]:
count_lemmatized_phrases_pipeline(lemmatized_groups, new_articles_2408_25, new_details_2408_25)

Successfully processed 27241 new phrase metrics and updated manifest.json.


'top_phrases_overtime.csv'